# Análise Financeira com Python — ClearBank

**Cenário:** analista de dados júnior em uma fintech. O objetivo é ler e validar um CSV de transações bancárias, calcular métricas financeiras mensais, sinalizar movimentações suspeitas e exportar o resultado em JSON.

**Estrutura do notebook:**
- Célula 1 — Imports e constantes
- Célula 2 — Funções utilitárias (`fmt_brl`, `ler_transacoes`, `validar_transacao`)
- Célula 3 — Funções de análise (`gerar_relatorio`, `exibir_relatorio`, `salvar_json`)
- Célula 4 — **Execução principal**
- Célula 5 — Opcional RO1: análise com pandas
- Célula 6 — Opcional RO2: gráfico com matplotlib


In [1]:
import csv
import json
from datetime import datetime

LIMITE_SUSPEITO = 10000.00
ARQUIVO_CSV = "transacoes.csv"
ARQUIVO_JSON = "relatorio.json"


In [2]:
def fmt_brl(valor):
    """Formata um número float no padrão monetário brasileiro: R$ 1.234,56"""
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def ler_transacoes(caminho):
    """Lê o arquivo CSV e retorna a lista de linhas brutas."""
    try:
        with open(caminho, newline="", encoding="utf-8") as f:
            leitor = csv.DictReader(f)
            return list(leitor)
    except FileNotFoundError:
        print(f"Erro: arquivo '{caminho}' não encontrado.")
        return []


def validar_transacao(linha):
    """
    Valida uma linha do CSV e retorna o registro limpo ou None se inválido.
    Descarta silenciosamente linhas com:
      - id vazio ou não numérico
      - cliente_id vazio
      - data fora do formato AAAA-MM-DD
      - tipo diferente de 'credito' ou 'debito'
      - valor não numérico ou <= 0
    """
    id_raw = linha.get("id", "").strip()
    if not id_raw:
        return None
    try:
        id_val = int(id_raw)
    except ValueError:
        return None

    cliente_id = linha.get("cliente_id", "").strip()
    if not cliente_id:
        return None

    data_raw = linha.get("data", "").strip()
    try:
        data = datetime.strptime(data_raw, "%Y-%m-%d")
    except ValueError:
        return None

    tipo = linha.get("tipo", "").strip().lower()
    if tipo not in ("credito", "debito"):
        return None

    valor_raw = linha.get("valor", "").strip()
    try:
        valor = float(valor_raw)
        if valor <= 0:
            return None
    except ValueError:
        return None

    return {
        "id": id_val,
        "data": data,
        "mes": data.strftime("%Y-%m"),
        "cliente_id": cliente_id,
        "tipo": tipo,
        "valor": valor,
        "descricao": linha.get("descricao", "").strip(),
        "categoria": linha.get("categoria", "").strip(),
        "suspeita": valor > LIMITE_SUSPEITO,
    }


In [3]:
def gerar_relatorio(transacoes_validas):
    """
    Agrupa as transações por mês e calcula métricas financeiras.
    Retorna dicionário com resumo mensal, período e transações suspeitas.
    """
    meses = {}
    for t in transacoes_validas:
        mes = t["mes"]
        if mes not in meses:
            meses[mes] = []
        meses[mes].append(t)

    resumo_mensal = {}
    for mes in sorted(meses):
        lista = meses[mes]
        creditos = [t["valor"] for t in lista if t["tipo"] == "credito"]
        debitos = [t["valor"] for t in lista if t["tipo"] == "debito"]
        total_credito = sum(creditos)
        total_debito = sum(debitos)
        valores = [t["valor"] for t in lista]
        resumo_mensal[mes] = {
            "quantidade": len(lista),
            "total_credito": round(total_credito, 2),
            "total_debito": round(total_debito, 2),
            "saldo": round(total_credito - total_debito, 2),
            "media": round(sum(valores) / len(valores), 2),
            "maior_valor": max(valores),
            "menor_valor": min(valores),
        }

    datas = [t["data"] for t in transacoes_validas]
    suspeitas = [t for t in transacoes_validas if t["suspeita"]]

    return {
        "resumo_mensal": resumo_mensal,
        "data_mais_antiga": min(datas),
        "data_mais_recente": max(datas),
        "dias_periodo": (max(datas) - min(datas)).days,
        "suspeitas": suspeitas,
    }


def exibir_relatorio(relatorio, total_lidas, total_validas, total_invalidas):
    """Exibe o relatório formatado no terminal."""
    print("\n" + "=" * 50)
    print(" RELATÓRIO FINANCEIRO — ClearBank")
    print("=" * 50)
    print(f"\nPERÍODO ANALISADO")
    print(f"  De:            {relatorio['data_mais_antiga'].strftime('%d/%m/%Y')}")
    print(f"  Até:           {relatorio['data_mais_recente'].strftime('%d/%m/%Y')}")
    print(f"  Total de dias: {relatorio['dias_periodo']}")
    print(f"\nTRANSAÇÕES")
    print(f"  Total lidas:   {total_lidas}")
    print(f"  Válidas:       {total_validas}")
    print(f"  Inválidas:     {total_invalidas}")

    print("\n" + "=" * 50)
    print(" RELATÓRIO MENSAL")
    print("=" * 50)
    for mes, dados in relatorio["resumo_mensal"].items():
        print(f"\nMês: {mes}")
        print(f"  Transações:    {dados['quantidade']}")
        print(f"  Total crédito: {fmt_brl(dados['total_credito'])}")
        print(f"  Total débito:  {fmt_brl(dados['total_debito'])}")
        print(f"  Saldo:         {fmt_brl(dados['saldo'])}")
        print(f"  Média:         {fmt_brl(dados['media'])}")
        print(f"  Maior valor:   {fmt_brl(dados['maior_valor'])}")
        print(f"  Menor valor:   {fmt_brl(dados['menor_valor'])}")

    print("\n" + "=" * 50)
    print(" TRANSAÇÕES SUSPEITAS")
    print("=" * 50)
    if relatorio["suspeitas"]:
        for s in relatorio["suspeitas"]:
            print(
                f"  ID: {s['id']} | Cliente: {s['cliente_id']} "
                f"| Data: {s['data'].strftime('%Y-%m-%d')} "
                f"| Valor: {fmt_brl(s['valor'])}"
            )
    else:
        print("  Nenhuma transação suspeita encontrada.")
    print()


def salvar_json(relatorio, total_validas, total_invalidas, caminho):
    """Salva o relatório completo em arquivo JSON."""
    suspeitas_serial = [
        {
            "id": s["id"],
            "cliente_id": s["cliente_id"],
            "data": s["data"].strftime("%Y-%m-%d"),
            "valor": s["valor"],
        }
        for s in relatorio["suspeitas"]
    ]

    saida = {
        "gerado_em": datetime.now().strftime("%Y-%m-%d"),
        "total_transacoes_validas": total_validas,
        "total_transacoes_invalidas": total_invalidas,
        "periodo": {
            "inicio": relatorio["data_mais_antiga"].strftime("%Y-%m-%d"),
            "fim": relatorio["data_mais_recente"].strftime("%Y-%m-%d"),
            "dias": relatorio["dias_periodo"],
        },
        "resumo_mensal": relatorio["resumo_mensal"],
        "transacoes_suspeitas": suspeitas_serial,
    }

    try:
        with open(caminho, "w", encoding="utf-8") as f:
            json.dump(saida, f, ensure_ascii=False, indent=2)
        print(f"Relatório salvo em '{caminho}'.")
    except Exception as e:
        print(f"Erro ao salvar JSON: {e}")


In [4]:
# ── Execução principal ────────────────────────────────────────────────────────

linhas_brutas = ler_transacoes(ARQUIVO_CSV)
total_lidas = len(linhas_brutas)

transacoes_validas = []
total_invalidas = 0

for linha in linhas_brutas:
    resultado = validar_transacao(linha)
    if resultado:
        transacoes_validas.append(resultado)
    else:
        total_invalidas += 1

total_validas = len(transacoes_validas)

print(f"Total de linhas lidas: {total_lidas}")
print(f"Linhas válidas:        {total_validas}")
print(f"Linhas inválidas:      {total_invalidas}")

relatorio = gerar_relatorio(transacoes_validas)
exibir_relatorio(relatorio, total_lidas, total_validas, total_invalidas)
salvar_json(relatorio, total_validas, total_invalidas, ARQUIVO_JSON)


Total de linhas lidas: 22
Linhas válidas:        17
Linhas inválidas:      5

 RELATÓRIO FINANCEIRO — ClearBank

PERÍODO ANALISADO
  De:            05/01/2026
  Até:           25/04/2026
  Total de dias: 110

TRANSAÇÕES
  Total lidas:   22
  Válidas:       17
  Inválidas:     5

 RELATÓRIO MENSAL

Mês: 2026-01
  Transações:    5
  Total crédito: R$ 4.700,00
  Total débito:  R$ 590,40
  Saldo:         R$ 4.109,60
  Média:         R$ 1.058,08
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 89,90

Mês: 2026-02
  Transações:    4
  Total crédito: R$ 18.500,00
  Total débito:  R$ 700,00
  Saldo:         R$ 17.800,00
  Média:         R$ 4.800,00
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 250,00

Mês: 2026-03
  Transações:    4
  Total crédito: R$ 16.000,00
  Total débito:  R$ 699,90
  Saldo:         R$ 15.300,10
  Média:         R$ 4.174,98
  Maior valor:   R$ 12.500,00
  Menor valor:   R$ 99,90

Mês: 2026-04
  Transações:    4
  Total crédito: R$ 3.500,00
  Total débito:  R$ 2.030,5

## Requisito Opcional RO1 — Análise com pandas

Reimplementação da leitura e do agrupamento mensal usando `pandas`. Os resultados devem ser idênticos aos da solução nativa.

In [5]:
import pandas as pd

df = pd.read_csv(ARQUIVO_CSV, dtype=str)
total_lidas_pd = len(df)

df = df.dropna(subset=["id", "cliente_id"])
df = df[df["id"].str.strip() != ""]
df = df[df["cliente_id"].str.strip() != ""]
df = df[df["tipo"].str.strip().str.lower().isin(["credito", "debito"])]
df["valor"] = pd.to_numeric(df["valor"], errors="coerce")
df = df[df["valor"].notna() & (df["valor"] > 0)]
df["data"] = pd.to_datetime(df["data"], format="%Y-%m-%d", errors="coerce")
df = df[df["data"].notna()]

total_validas_pd = len(df)
total_invalidas_pd = total_lidas_pd - total_validas_pd
df["mes"] = df["data"].dt.strftime("%Y-%m")
df["tipo"] = df["tipo"].str.strip().str.lower()

creditos_pd = df[df["tipo"] == "credito"].groupby("mes")["valor"].sum().rename("total_credito")
debitos_pd  = df[df["tipo"] == "debito"].groupby("mes")["valor"].sum().rename("total_debito")
qtd_pd      = df.groupby("mes")["valor"].count().rename("quantidade")
media_pd    = df.groupby("mes")["valor"].mean().rename("media")
maximo_pd   = df.groupby("mes")["valor"].max().rename("maior_valor")
minimo_pd   = df.groupby("mes")["valor"].min().rename("menor_valor")

resumo_pd = pd.concat([qtd_pd, creditos_pd, debitos_pd, media_pd, maximo_pd, minimo_pd], axis=1).fillna(0)
resumo_pd["saldo"] = resumo_pd["total_credito"] - resumo_pd["total_debito"]
resumo_pd = resumo_pd.sort_index()

print("=" * 50)
print(" ANÁLISE COM PANDAS — ClearBank")
print("=" * 50)
print(f"\nTotal lidas:   {total_lidas_pd}")
print(f"Válidas:       {total_validas_pd}")
print(f"Inválidas:     {total_invalidas_pd}")

print("\n" + "=" * 50)
print(" RESUMO MENSAL (pandas)")
print("=" * 50)
for mes, row in resumo_pd.iterrows():
    print(f"\nMês: {mes}")
    print(f"  Transações:    {int(row['quantidade'])}")
    print(f"  Total crédito: {fmt_brl(row['total_credito'])}")
    print(f"  Total débito:  {fmt_brl(row['total_debito'])}")
    print(f"  Saldo:         {fmt_brl(row['saldo'])}")
    print(f"  Média:         {fmt_brl(row['media'])}")
    print(f"  Maior valor:   {fmt_brl(row['maior_valor'])}")
    print(f"  Menor valor:   {fmt_brl(row['menor_valor'])}")

print("\n" + "=" * 50)
print(" COMPARAÇÃO COM SOLUÇÃO NATIVA")
print("=" * 50)
referencia = {
    "2026-01": {"total_credito": 4700.00, "total_debito": 590.40, "saldo": 4109.60},
    "2026-02": {"total_credito": 18500.00, "total_debito": 700.00, "saldo": 17800.00},
    "2026-03": {"total_credito": 16000.00, "total_debito": 699.90, "saldo": 15300.10},
    "2026-04": {"total_credito": 3500.00, "total_debito": 2030.50, "saldo": 1469.50},
}
todos_iguais = True
for mes, ref in referencia.items():
    row = resumo_pd.loc[mes]
    ok = (
        round(row["total_credito"], 2) == ref["total_credito"]
        and round(row["total_debito"], 2) == ref["total_debito"]
        and round(row["saldo"], 2) == ref["saldo"]
    )
    print(f"  {mes}: {'✓ OK' if ok else '✗ DIVERGENTE'}")
    if not ok:
        todos_iguais = False
print("\nResultados idênticos à solução nativa." if todos_iguais else "\nAtenção: há divergências.")


 ANÁLISE COM PANDAS — ClearBank

Total lidas:   22
Válidas:       17
Inválidas:     5

 RESUMO MENSAL (pandas)

Mês: 2026-01
  Transações:    5
  Total crédito: R$ 4.700,00
  Total débito:  R$ 590,40
  Saldo:         R$ 4.109,60
  Média:         R$ 1.058,08
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 89,90

Mês: 2026-02
  Transações:    4
  Total crédito: R$ 18.500,00
  Total débito:  R$ 700,00
  Saldo:         R$ 17.800,00
  Média:         R$ 4.800,00
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 250,00

Mês: 2026-03
  Transações:    4
  Total crédito: R$ 16.000,00
  Total débito:  R$ 699,90
  Saldo:         R$ 15.300,10
  Média:         R$ 4.174,98
  Maior valor:   R$ 12.500,00
  Menor valor:   R$ 99,90

Mês: 2026-04
  Transações:    4
  Total crédito: R$ 3.500,00
  Total débito:  R$ 2.030,50
  Saldo:         R$ 1.469,50
  Média:         R$ 1.382,62
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 180,50

 COMPARAÇÃO COM SOLUÇÃO NATIVA
  2026-01: ✓ OK
  2026-02: ✓ OK
  2026-

## Requisito Opcional RO2 — Gráfico com matplotlib

Gráfico de barras agrupadas com crédito e débito por mês, salvo como `grafico.png`.

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

meses_graf = list(relatorio["resumo_mensal"].keys())
creditos_graf = [relatorio["resumo_mensal"][m]["total_credito"] for m in meses_graf]
debitos_graf  = [relatorio["resumo_mensal"][m]["total_debito"]  for m in meses_graf]

x = np.arange(len(meses_graf))
largura = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
barras_c = ax.bar(x - largura / 2, creditos_graf, largura, label="Crédito", color="#2ecc71", edgecolor="white")
barras_d = ax.bar(x + largura / 2, debitos_graf,  largura, label="Débito",  color="#e74c3c", edgecolor="white")

for barra, cor in [(barras_c, "#2ecc71"), (barras_d, "#e74c3c")]:
    for b in barra:
        v = b.get_height()
        ax.text(b.get_x() + b.get_width() / 2, v + 150,
                f"R$ {v:,.0f}".replace(",", "."),
                ha="center", va="bottom", fontsize=8, color=cor, fontweight="bold")

ax.set_title("Crédito e Débito por Mês — ClearBank 2026", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Mês", fontsize=11)
ax.set_ylabel("Valor (R$)", fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(meses_graf, fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"R$ {v:,.0f}".replace(",", ".")))
ax.legend(fontsize=10)
ax.set_facecolor("#f9f9f9")
fig.patch.set_facecolor("#ffffff")
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig("grafico.png", dpi=150, bbox_inches="tight")
print("Gráfico salvo em 'grafico.png'.")
plt.show()


Gráfico salvo em 'grafico.png'.
